## Impact of Program Design

Writing a program that runs is not the same as writing a program that works. A method can pass every test you throw at it and still be wrong the moment someone hands it an input you did not imagine -- and once it ships, that same code affects real people, in ways both good and bad, that its author never intended.

### Learning Targets

- I can explain what system reliability means and why passing one test is not evidence of it.
- I can add validation that rejects invalid input before it corrupts an object's state.
- I can describe both beneficial and harmful effects of the same program, and explain how a program can cause harm beyond its intended use.
- I can decide whether code I found online may be reused, based on its license.

## Lesson Design: The LxD Cycle

This lesson was built with the Learning Experience Design cycle -- every choice below traces to something observed about actual learners.

### Empathize

Watching classmates write their first validated methods, the same pattern showed up repeatedly: they tested one input, it returned the expected value, and they moved on. When asked "what happens if someone passes a negative number," the common answer was some version of *"why would they?"*

The misconception underneath it: **invalid input is something users do wrong, not something the method is responsible for handling.** Testing is treated as confirmation that the code works, not as an attempt to break it.

This is worth teaching because it produces no error. The code compiles, one test passes, and the failure surfaces later in a context where nobody is looking for it.

### Define

**Point of View:** A CSA student who can write a working method needs to see that a method owns its own invalid input -- they currently test to confirm success, not to find failure, which hides silent corruption.

### Ideate

**How Might We:** make a silent failure *visible* to someone whose code has never errored? Chosen activity: a trace table with a deliberately wrong row, so students watch a method accept -400 degrees and confidently report a status. `Thermostat` was chosen because an impossible value needs no domain knowledge to spot, and it scales into the impact sections below.

### Prototype & Test

The worked example, the broken-then-fixed pair, and the practice tasks below are that prototype -- taught and refined on the assigned teaching day.

## The Running Example

**ClimateSense** is a hypothetical connected thermostat product, the kind installed in tens of thousands of homes to control heating and cooling remotely. Every example below is built around one piece of it: a `Thermostat` class that holds a target temperature and reports what the system should currently be doing -- heating, cooling, or idle. Keep that product framing in mind; the same class you're about to see is what a real device like this would run, and the failures it can hide are the same ones a shipped product would have.

## Part 1: System Reliability

**System reliability** means a program performs as expected, under stated conditions, without failure -- not just for the one input you happened to try. The class below is `ClimateSense`'s unguarded first version. Predict all three outputs before reading past the cell.

In [ ]:
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        targetTemp = temp;
    }

    public double getTargetTemp() {
        return targetTemp;
    }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }
}

Thermostat t = new Thermostat("Lab", 72);
System.out.println(t.getStatus());

t.setTargetTemp(80);
System.out.println(t.getStatus());

t.setTargetTemp(-400);          // below absolute zero
System.out.println(t.getStatus());

| Call | `targetTemp` after | `getStatus()` | Correct? |
|---|---|---|---|
| `new Thermostat("Lab", 72)` | 72.0 | `IDLE` | Yes |
| `setTargetTemp(80)` | 80.0 | `COOLING` | Yes |
| `setTargetTemp(-400)` | -400.0 | `HEATING` | **No** -- below absolute zero |

Nothing crashed. Nothing warned. The program accepted an impossible temperature and confidently reported a status for it. A silent failure is worse than a crash, because a crash at least tells you where to look.

### Popcorn Hacks

1. `getStatus()` uses `>= 78` and `<= 65`. What does it return for exactly 78? For exactly 65?

## Part 2: Guarding Against Bad Input

A **guard clause** is a check placed at the very top of a method that rejects invalid input immediately, before anything else happens. `setTargetTemp` above has no guard clause at all -- `-400`, `9999`, even `Double.NaN` all succeed silently. The version below adds one, validating before it assigns:

In [ ]:
public class Thermostat {
    private String roomName;
    private double targetTemp;

    public Thermostat(String roomName, double targetTemp) {
        this.roomName = roomName;
        this.targetTemp = targetTemp;
    }

    public void setTargetTemp(double temp) {
        if (Double.isNaN(temp) || temp < 50 || temp > 90) {
            throw new IllegalArgumentException("Target temp out of range: " + temp);
        }
        targetTemp = temp;
    }

    public double getTargetTemp() { return targetTemp; }

    public String getStatus() {
        if (targetTemp >= 78) return "COOLING";
        if (targetTemp <= 65) return "HEATING";
        return "IDLE";
    }
}

Thermostat t = new Thermostat("Lab", 72);
try {
    t.setTargetTemp(-400);
} catch (IllegalArgumentException e) {
    System.out.println("rejected -> " + e.getMessage());
}
System.out.println(t.getTargetTemp());   // still 72.0 -- the exception ran before the assignment

The check runs first, so a rejected call leaves the object exactly as it was -- a method that *fails* is safer than one that *corrupts*. `Double.isNaN(temp)` needs its own clause because `NaN` fails every comparison: `NaN < 50` and `NaN > 90` are both `false`, so a range check alone would let it slip through.

### Popcorn Hacks

1. Delete the `Double.isNaN(temp)` clause, predict what `setTargetTemp(Double.NaN)` does now, then explain why in one sentence.

## Part 3: Social, Economic, and Cultural Impact

The same program can be beneficial and harmful at once -- not a balance where one cancels the other. Take ClimateSense into fifty thousand connected homes:

| Effect | Beneficial | Harmful |
|---|---|---|
| Energy use | Lower bills, less grid strain | -- |
| Auto scheduling | Comfort without manual adjustment | Penalizes shift workers |
| Remote access | Adjust the house before arriving | Needs a smartphone and home internet |

None of the harms are bugs -- they're consequences of reasonable design choices. A designer is responsible for asking who's excluded, not just who's served.

## Part 4: Unintended Consequences

An **unintended consequence** is harm that shows up beyond a program's intended use, even when the code behind it works exactly as designed. Suppose ClimateSense nudges every target temp up two degrees during a heat wave, to ease grid load. For one house, barely noticeable. For fifty thousand houses acting at once, it's a demand shock the grid wasn't built for -- and for a household with a medically vulnerable resident, two degrees is not a nudge.

Nobody wrote a harmful feature -- it does exactly what it was designed to do. The harm comes from scale, not intent.

## Part 5: Intellectual Property and Code Reuse

Reusing other people's code is normal -- the rules are about *which* code, under *what terms*.

| What you found | May you use it? |
|---|---|
| Open source, license stated | Yes -- follow the license |
| Not open source | Only with permission |
| No license stated | Treat it as **not** free to use |

**Being able to read code isn't being allowed to use it.** A public repo is visible to everyone; that says nothing about permission. No license means write your own -- "it was on the internet" is not a defense.

### Popcorn Hacks

1. A classmate says a four-line snippet is too short for licensing to apply. Respond.

## Optional Extension

Permissive open source licenses allow reuse in closed-source projects; copyleft licenses require your own code to be published under the same terms too.

## Practice: Trace and Debug

### Predict the Output

Using the **guarded** `setTargetTemp` above:

```java
Thermostat t = new Thermostat("Lab", 72);
try {
    t.setTargetTemp(95);
} catch (IllegalArgumentException e) {
    System.out.println("rejected");
}
System.out.println(t.getTargetTemp());
System.out.println(t.getStatus());
```

A. `95.0`, then `COOLING`

B. `rejected`, then `72.0`, then `IDLE`

C. `rejected`, then `95.0`, then `COOLING`

D. `rejected`, then `72.0`, then `COOLING`

### Apply the Idea

`Room` needs the same guard-clause treatment `Thermostat` just got. Finish `setRoomName` below so it rejects `null`, empty, and whitespace-only names -- validate before assigning, the same way `setTargetTemp` does. The first attempt should be accepted and the next three rejected.

In [ ]:
public class Room {
    private String roomName;

    public Room(String roomName) { this.roomName = roomName; }

    public void setRoomName(String name) {
        // TODO: reject null, empty, and whitespace-only names.
        roomName = name;
    }

    public String getRoomName() { return roomName; }
}

Room r = new Room("Lab");
String[] attempts = { "Nursery", null, "", "   " };
for (String a : attempts) {
    try {
        r.setRoomName(a);
        System.out.println("accepted -> [" + r.getRoomName() + "]");
    } catch (IllegalArgumentException e) {
        System.out.println("rejected -> " + e.getMessage());
    }
}

## Answer Check

**Predict the Output -- B.** 95 is outside 50-90, so the guard throws before the assignment runs -- the object still holds 72.0, and `getStatus()` returns `IDLE`. (A and C wrongly assume the value was stored anyway; D mismatches its own state.)

**Apply the Idea.** Reject `null` -- a later `String` method call on it would throw `NullPointerException` far from the real cause -- and reject empty or whitespace-only names, since an unidentifiable room is unusable. The rule: a mutator should reject anything that leaves the object in a state it can't function in.

## Quick Review

- **System reliability** means working under all stated conditions, not just the one input you tried.
- Test edges: zero, negative, empty, `null`, `NaN`, and boundary values.
- A **guard clause** that runs before assignment stops invalid input from ever being stored.
- A method that **silently accepts invalid input** is more dangerous than one that crashes.
- The same program can be **both beneficial and harmful**, and can cause harm **beyond its intended use** -- usually from scale, not bad intent.
- **Publicly visible code is not free to use.** No license stated means no permission granted.

## Sources

- College Board, *AP Computer Science A Course and Exam Description*, Unit 3, Topic 3.2.
- Oracle, ["Class Double"](https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/Double.html) -- `isNaN` and `NaN` comparison behavior.
- Oracle, ["Class IllegalArgumentException"](https://docs.oracle.com/en/java/javase/17/docs/api/java.base/java/lang/IllegalArgumentException.html).
- Open Source Initiative, ["The Open Source Definition"](https://opensource.org/osd).
- GitHub Docs, ["Licensing a repository"](https://docs.github.com/en/repositories/managing-your-repositorys-settings-and-features/customizing-your-repository/licensing-a-repository).